## 30.4) Biblioteca PyOD => detecção de outliers

- Documentação: https://pyod.readthedocs.io/en/latest/#

(i) Individual Detection Algorithms :



Categoria	|	Algoritmos Famosos	|	Como funciona (Resumo)	|

Proximidade	|	kNN, LOF, COF	|	Medem a distância ou densidade local. Se um ponto está muito longe dos vizinhos ou em uma zona de baixa densidade, é um outlier.	|

Lineares	|	PCA, MCD, OCSVM	|	Usam projeções lineares. No PCA, por exemplo, outliers são pontos que não podem ser bem reconstruídos usando apenas os componentes principais.	|

Probabilísticos	|	ECOD, COPOD, GMM	|	Calculam a probabilidade de um ponto pertencer à distribuição dos dados. O ECOD é extremamente rápido e baseado em funções de distribuição acumulada.	|

Baseados em Árvores	|	Isolation Forest, LODA	|	O Isolation Forest "isola" as observações. Como outliers são raros e diferentes, eles são isolados com menos divisões na árvore do que pontos normais.	|

Redes Neurais	|	AutoEncoder, VAE, GAN	|	Tentam comprimir e reconstruir os dados. Outliers geralmente têm um erro de reconstrução muito alto.	|

- Neste exemplo usaremos o KNN, de proximidade

ambiente anaconda Udemy => erro no KNN ImportError: Numba needs NumPy 2.2 or less. Got NumPy 2.3.

ambiente anaconda orange3 => deu certo


In [10]:
#!pip install pyod

### importar o algoritmo de interesse

In [11]:
from pyod.models.knn import KNN

In [12]:
import numpy as np
print(np.__version__) # orange3

1.26.4


In [13]:
# versão udemy
#2.3.4


In [14]:
import pandas as pd
base_credit = pd.read_csv('credit_data.csv')
base_credit.tail()

,clientid,income,age,loan,default
1995,1996,59221.044874,48.518179,1926.729397,0
1996,1997,69516.127573,23.162104,3503.176156,0
1997,1998,44311.449262,28.017167,5522.786693,1
1998,1999,43756.056605,63.971796,1622.722598,0
1999,2000,69436.579552,56.152617,7378.833599,0


In [15]:
base_credit.head(5)

,clientid,income,age,loan,default
0,1,66155.925095,59.017015,8106.532131,0
1,2,34415.153966,48.117153,6564.745018,0
2,3,57317.170063,63.108049,8020.953296,0
3,4,42709.534201,45.751972,6103.642260,0
4,5,66952.688845,18.584336,8770.099235,1


In [17]:
detector = KNN()

In [19]:
detector.get_params()

{'algorithm': 'auto',
 'contamination': 0.1,
 'leaf_size': 30,
 'method': 'largest',
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': 1,
 'n_neighbors': 5,
 'p': 2,
 'radius': 1.0}

contaminação base da distância

metric: Minkowski

 'p': 2, dstância euclidiana


In [21]:
base_credit.columns[1:4] # para ver o outliers serão essas colunas

Index(['income', 'age', 'loan'], dtype='object')

In [23]:
# Verifica a contagem de nulos em cada coluna
print(base_credit.iloc[:, 1:4].isnull().sum()) # deu erro no fit pois havia dados null

income    0
age       3
loan      0
dtype: int64


In [24]:
# Seleciona as colunas que você vai usar
X_credit = base_credit.iloc[:, 1:4]

# Preenche os nulos com a mediana de cada coluna
X_credit = X_credit.fillna(X_credit.median())

# Agora o fit deve funcionar normalmente
detector.fit(X_credit)

,contamination,0.1
,n_neighbors,5
,method,'largest'
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'minkowski'
,p,2
,metric_params,None
,n_jobs,1


In [25]:

# fit para fazer o treinamento
# calculo do treinamento com distância KNN
# os dados que tiverem as maiores distâncias serão considerados outliers
#detector.fit(base_credit.iloc[:,1:4]) #coletar todos os dados

In [26]:
detector.get_params()

{'algorithm': 'auto',
 'contamination': 0.1,
 'leaf_size': 30,
 'method': 'largest',
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': 1,
 'n_neighbors': 5,
 'p': 2,
 'radius': 1.0}

### 0 não é outliers e 1 é outliers

In [27]:
previsoes = detector.labels_
previsoes

array([0, 0, 0, ..., 0, 0, 1])


Esses são os hiperparâmetros do modelo KNN (K-Nearest Neighbors) do PyOD. Eles definem as "regras de negócio" que o algoritmo usará para decidir quem é um dado normal e quem é um outlier.



1. Os "Corações" do Modelo (Mais Importantes)
 
- contamination (0.1): É a porcentagem de outliers que você espera encontrar. 0.1 significa que o modelo vai marcar os 10% dos dados com as maiores distâncias como anomalias.

- n_neighbors (5): É o valor de $k$. O algoritmo olhará para os 5 vizinhos mais próximos de cada ponto para calcular o quão isolado ele está.

- method ('largest'): Define como o "score de anomalia" é calculado.

a) largest: Usa a distância até o 5º vizinho mais distante.

b) mean: Usa a média das distâncias de todos os 5 vizinhos.

- metric ('minkowski') e p (2): Definem a fórmula matemática da distância. Quando p=2, usando a Distância Euclidiana (a famosa linha reta entre dois pontos).



2. Configurações de Performance (Velocidade)
   
- algorithm ('auto'): O PyOD escolhe automaticamente a melhor estrutura de dados para buscar os vizinhos (como KDTree ou BallTree), dependendo do tamanho do seu dataset.

- leaf_size (30): É um parâmetro interno das árvores de busca (KDTree/BallTree). Afeta a velocidade da busca e o uso de memória, mas não o resultado final.

- n_jobs (1): Quantos núcleos do seu processador ele vai usar. 1 significa que ele usará apenas um núcleo. Para datasets gigantes, usar -1 aceleraria o processo usando todos os núcleos.

3. Parâmetros Específicos

- radius (1.0): Usado apenas se estivesse fazendo uma busca por raio em vez de número fixo de vizinhos (não é o caso aqui).

- metric_params (None): Serve para passar configurações extras para métricas de distância muito complexas ou personalizadas.



"Vou olhar para os 5 vizinhos mais próximos de cada registro usando a distância em linha reta. Se a distância até o 5º vizinho for muito grande (comparada aos outros), vou marcar esse registro como um dos 10% de outliers do meu banco de dados."

### contagens de quantos dados são de 0 ou 1

In [29]:
np.unique(previsoes, return_counts=True)

(array([0, 1]), array([1800,  200]))

In [ ]:
# valores sem ser outliers => 0

In [31]:
np.unique(previsoes, return_counts=True)[0][0]

0

In [34]:
print(f'Não é outliers {np.unique(previsoes, return_counts=True)[0][0]} e sua quantidade é de {np.unique(previsoes, return_counts=True)[1][0]}.')

Não é outliers 0 e sua quantidade é de 1800.


In [35]:
print(f'É outliers {np.unique(previsoes, return_counts=True)[0][1]} e sua quantidade é de {np.unique(previsoes, return_counts=True)[1][1]}.')

É outliers 1 e sua quantidade é de 200.


cálculo da distância com todos os atributos, diferentemente do gráfico do boxplot, ou do dispersão, os números de atributos são limitados, neste caso não.

Verificar a confiança dos dados

In [36]:
confianca_previsoes = detector.decision_scores_
confianca_previsoes

array([ 704.78948078,  365.218309  ,  583.2159934 , ...,  395.01466508,
        557.88978241, 1071.5109404 ])

confiança muito alta indica uma distância muito grande

In [41]:
previsoes[-4:] # últimos 4 dados

array([1, 0, 0, 1])

In [42]:
confianca_previsoes[-4:]

array([1015.87339884,  395.01466508,  557.88978241, 1071.5109404 ])

# verificar se é outliers ou não

In [46]:
outliers = []
for i in range(len(previsoes)):
  print(f'{i}) previsôes: {previsoes[i]} e confiança previsões: {confianca_previsoes[i]}')
  if previsoes[i] == 1:
    outliers.append(i)

0) previsôes: 0 e confiança previsões: 704.7894807791696
1) previsôes: 0 e confiança previsões: 365.21830899985906
2) previsôes: 0 e confiança previsões: 583.2159933969901
3) previsôes: 0 e confiança previsões: 669.1172307418985
4) previsôes: 1 e confiança previsões: 952.6002186476792
5) previsôes: 0 e confiança previsões: 546.2634915427162
6) previsôes: 0 e confiança previsões: 347.1669481891248
7) previsôes: 0 e confiança previsões: 573.3947097459896
8) previsôes: 0 e confiança previsões: 532.7286552405403
9) previsôes: 0 e confiança previsões: 561.8260615496696
10) previsôes: 0 e confiança previsões: 611.0671075771143
11) previsôes: 0 e confiança previsões: 645.8081658751695
12) previsôes: 0 e confiança previsões: 540.9591139816707
13) previsôes: 0 e confiança previsões: 611.5593802198417
14) previsôes: 0 e confiança previsões: 814.334639739793
15) previsôes: 0 e confiança previsões: 723.1341876580393
16) previsôes: 0 e confiança previsões: 679.8718686007359
17) previsôes: 0 e confi

In [47]:
outliers

[4,
 24,
 29,
 34,
 38,
 78,
 90,
 95,
 104,
 105,
 111,
 115,
 128,
 157,
 160,
 163,
 165,
 186,
 218,
 220,
 234,
 237,
 275,
 282,
 292,
 297,
 304,
 324,
 325,
 335,
 342,
 346,
 350,
 360,
 361,
 394,
 402,
 404,
 405,
 414,
 421,
 422,
 427,
 449,
 450,
 452,
 454,
 480,
 488,
 489,
 505,
 508,
 523,
 531,
 533,
 535,
 548,
 573,
 599,
 633,
 646,
 663,
 697,
 710,
 716,
 737,
 748,
 765,
 767,
 842,
 851,
 869,
 878,
 880,
 885,
 898,
 901,
 927,
 930,
 933,
 940,
 945,
 946,
 952,
 975,
 977,
 990,
 992,
 996,
 999,
 1002,
 1006,
 1018,
 1025,
 1028,
 1034,
 1036,
 1040,
 1041,
 1050,
 1057,
 1065,
 1083,
 1088,
 1091,
 1092,
 1094,
 1101,
 1102,
 1128,
 1139,
 1140,
 1184,
 1197,
 1200,
 1202,
 1208,
 1210,
 1212,
 1219,
 1222,
 1235,
 1236,
 1239,
 1291,
 1292,
 1303,
 1338,
 1350,
 1351,
 1370,
 1377,
 1378,
 1379,
 1382,
 1385,
 1388,
 1389,
 1396,
 1414,
 1415,
 1417,
 1428,
 1429,
 1435,
 1443,
 1448,
 1454,
 1470,
 1481,
 1492,
 1506,
 1514,
 1516,
 1520,
 1566,
 1582,


In [48]:
print('lista dos ids que são outliers')
print(outliers)

lista dos ids que são outliers
[4, 24, 29, 34, 38, 78, 90, 95, 104, 105, 111, 115, 128, 157, 160, 163, 165, 186, 218, 220, 234, 237, 275, 282, 292, 297, 304, 324, 325, 335, 342, 346, 350, 360, 361, 394, 402, 404, 405, 414, 421, 422, 427, 449, 450, 452, 454, 480, 488, 489, 505, 508, 523, 531, 533, 535, 548, 573, 599, 633, 646, 663, 697, 710, 716, 737, 748, 765, 767, 842, 851, 869, 878, 880, 885, 898, 901, 927, 930, 933, 940, 945, 946, 952, 975, 977, 990, 992, 996, 999, 1002, 1006, 1018, 1025, 1028, 1034, 1036, 1040, 1041, 1050, 1057, 1065, 1083, 1088, 1091, 1092, 1094, 1101, 1102, 1128, 1139, 1140, 1184, 1197, 1200, 1202, 1208, 1210, 1212, 1219, 1222, 1235, 1236, 1239, 1291, 1292, 1303, 1338, 1350, 1351, 1370, 1377, 1378, 1379, 1382, 1385, 1388, 1389, 1396, 1414, 1415, 1417, 1428, 1429, 1435, 1443, 1448, 1454, 1470, 1481, 1492, 1506, 1514, 1516, 1520, 1566, 1582, 1584, 1608, 1610, 1616, 1618, 1626, 1630, 1635, 1648, 1660, 1668, 1672, 1675, 1679, 1681, 1694, 1705, 1716, 1720, 1727, 1733,

Registro completo dos clientes

In [49]:
#[:,:] # : todos os registros
# [outliers,:] indicar qual ids selecionados que são outliers
lista_outliers = base_credit.iloc[outliers,:]
lista_outliers

,clientid,income,age,loan,default
4,5,66952.688845,18.584336,8770.099235,1
24,25,65301.984029,48.840922,5465.267886,0
29,30,58842.891308,54.510948,10871.186790,0
34,35,57584.973790,36.672021,1728.423755,0
38,39,60921.063104,18.840526,968.836383,0
...,...,...,...,...,...
1943,1944,59792.508585,24.187499,660.241453,0
1944,1945,35879.519994,41.072935,5335.403499,0
1957,1958,50458.958203,52.314565,9852.889427,0
1996,1997,69516.127573,23.162104,3503.176156,0


In [50]:
# lista completa e salvar para a detecção de fraude
lista_outliers.tail(10)

,clientid,income,age,loan,default
1913,1914,69992.332712,41.771231,52.872190,0
1914,1915,68110.239953,32.171575,11029.667710,1
1915,1916,48015.554743,28.241796,54.008242,0
1930,1931,44241.283000,19.982539,8733.179295,1
1935,1936,33707.801036,59.917644,321.578931,0
1943,1944,59792.508585,24.187499,660.241453,0
1944,1945,35879.519994,41.072935,5335.403499,0
1957,1958,50458.958203,52.314565,9852.889427,0
1996,1997,69516.127573,23.162104,3503.176156,0
1999,2000,69436.579552,56.152617,7378.833599,0
